## Artificial Neural Network from Scratch with Pytorch

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [32]:
transforms = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root="./data",train = True,download=True,transform=transforms)
test_dataset = datasets.MNIST(root="./data",train=False,download=True,transform=transforms)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)


In [33]:
train_dataset, test_dataset

(Dataset MNIST
     Number of datapoints: 60000
     Root location: ./data
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
            ),
 Dataset MNIST
     Number of datapoints: 10000
     Root location: ./data
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
            ))

In [34]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten() # Flattens 28 * 28 image to a 784 vector
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128), # hidden Layer
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [35]:
#Intialize Model , loss, ans optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [36]:
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss() # Standard for multi class classification
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [37]:
# Training Loop
epochs = 5

print(f"Trainig pytorch model on {device}")

for epochs in range(epochs):
    model.train()
    running_loss = 0.0
    for X,y in train_loader:
        X,y = X.to(device),y.to(device)
        
        #Forward Pass
        pred = model(X)
        loss = criterion(pred,y)
        
        #Backpropagation 
        optimizer.zero_grad() #
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epochs+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f}")

Trainig pytorch model on cuda
Epoch 1/0 | Loss: 0.3448
Epoch 2/1 | Loss: 0.1582
Epoch 3/2 | Loss: 0.1105
Epoch 4/3 | Loss: 0.0826
Epoch 5/4 | Loss: 0.0656


In [38]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, predicted = torch.max(outputs.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 97.31%


In [39]:
accuracy = 100 - ((running_loss/len(train_loader))*100)


In [40]:
print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 93.44%
